In [ ]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from functools import partial

from folktexts.acs import ACSDataset, ACSTaskMetadata
from folktexts.llm_utils import load_model_tokenizer
from folktexts.classifier import TransformersLLMClassifier
from folktexts.prompting import encode_row_prompt_few_shot


In [4]:
task_name = 'ACSIncome'
task = ACSTaskMetadata.get_task(task_name)
acs_dataset = ACSDataset.make_from_task(
    task=task,
    cache_dir='./data',)

In [6]:
X_sample, _y_sample = acs_dataset.sample_n_train_examples(n=1)

In [5]:
model_name_or_path = "../models/meta-llama--Meta-Llama-3.2-1B-Instruct"
model, tokenizer = load_model_tokenizer(model_name_or_path)

In [13]:
n_shots = 3
llm_clf = TransformersLLMClassifier(
    model=model,
    tokenizer=tokenizer,
    task=task,
    batch_size=50,
    context_size=600,
    encode_row=partial(
        encode_row_prompt_few_shot,
        task=task,
        dataset=acs_dataset,
        n_shots=n_shots,
        class_balancing=True,
        reuse_examples=True,
        prompt_style={"format": "bullet", "connector": "is"},
    ),
)


print(llm_clf.encode_row(X_sample.iloc[0], question=llm_clf.task.question))

The following data corresponds to different survey respondents. The survey was conducted among US residents in 2018. Please answer each question based on the information provided. The data provided is enough to reach an approximate answer for each person.

Information:
- age is 43 years old
- class of worker is Working for a for-profit private company or organization
- highest educational attainment is Some college, 1 or more years, no degree
- marital status is Divorced
- occupation is Sales representatives of services, except advertising, insurance, financial services, and travel
- place of birth is Michigan
- relationship to the reference person in the survey is The reference person itself
- usual number of hours worked per week is 40 hours
- sex is Male
- race is Two or more races

Question: What is this person's estimated yearly income?
A. Below $50,000.
B. Above $50,000.
Answer: B

Information:
- age is 51 years old
- class of worker is Working for a for-profit private company or

In [ ]:
for n_shots in [4, 10, 50, 100]:
    llm_clf = TransformersLLMClassifier(
        model=model,
        tokenizer=tokenizer,
        task=task,
        encode_row=partial(
            encode_row_prompt_few_shot,
            task=task,
            dataset=acs_dataset,
            n_shots=n_shots,
            class_balancing=True,
            reuse_examples=True, # make sure these are always the same examples
            prompt_style={"format": "bullet", "connector": "is"},
        ),
    )

    num_reps = 50 
    repeated_sample = pd.concat([X_sample]*num_reps)
    predictions_repeated_sample = llm_clf.predict_proba(repeated_sample)[:,1] #returns multi-class probs
    
    plt.hist(predictions_repeated_sample, weights= np.zeros((num_reps, )) + 1./num_reps);plt.ylim(0,1)
    plt.title(f'{n_shots} shots')
    plt.show()